# RAREXT χ=48 rebuild — GPU training for the ledger `diff` family

This notebook runs the **same projected-training loop** the shipped artifact was
refined with — `tools/entangle/gpu_train_diff.py` — on a Colab GPU instead of
the 4-core Mac. Every batch contains 30–50% replay of the deployed skills, every
50 steps the weights are SVD-projected back onto the χ=48 entanglement
manifold, and the promotion gate is measured, not assumed: **no deployed skill
may regress by more than 0.07 on 24 held-out samples.**

1. **Runtime → Change runtime type → T4 GPU**, then **Run all**.
2. Cell 1 clones the protocol repo (the starting checkpoint
   `refined-chi48.pt` is the exact 4,673,760-byte rebuild the page serves).
3. Cell 2 trains and evaluates per digit width (1d/2d/3d/4d).
4. Cell 3 packs the artifacts; download them and drop them into
   `tools/entangle/` of your clone — then locally:

       node scripts/promote-entangled.mjs --candidate tools/entangle/entangled-chi48-diff-gpu.gguf
       node tools/rank/eval-neural.mjs --n 200 --seed 20261123


In [ ]:
import sys, torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("!! Runtime > Change runtime type > T4 GPU, then re-run !!")

if not __import__("os").path.isdir("/content/RAREXT-protocol"):
    !git clone --depth 1 https://github.com/999svs999/RAREXT-protocol.git /content/RAREXT-protocol

sys.path.insert(0, "/content/RAREXT-protocol/tools/entangle")
sys.argv = [
    "gpu_train_diff",
    "--steps", "20000",
    "--batch", "64",
    "--out", "/content",
    "--ckpt", "/content/RAREXT-protocol/tools/entangle/refined-chi48.pt",
]
import gpu_train_diff
gpu_train_diff.main()


In [ ]:
# pack the artifacts for the local promotion step
import shutil
shutil.make_archive("/content/rarext-gpu-diff", "zip", "/content",
                    base_dir="gpu-refined-diff.pt")
import os
from google.colab import files
for f in ("/content/gpu-refined-diff.pt", "/content/gpu-train.json"):
    print(f, os.path.getsize(f), "bytes")
# files.download("/content/gpu-refined-diff.pt")
# files.download("/content/gpu-train.json")
